# Chest X-ray Fairness Audit — Kaggle GPU run

Runs one training + audit pass on NIH ChestX-ray14. Baseline and mitigated are two
independent trainings (both start from ImageNet weights), so they are run as two
separate commits rather than one 12h+ session.

**Setup**
1. Add data: *Add Input* → search `nih-chest-xrays/data` → it mounts at `/kaggle/input/data`.
2. Settings → Accelerator → **GPU P100** (or T4), Internet → **On** (needed for the clone).
3. Set `RUN` below, then **Save & Run All (Commit)**. Interactive sessions are discarded;
   only a commit persists `/kaggle/working`.
4. When it finishes, run again with `RUN = "mitigated"`.

In [ ]:
RUN = "baseline"  # "baseline" | "mitigated"

In [ ]:
# Clone outside /kaggle/working so the commit output holds only checkpoints + results.
!rm -rf /tmp/repo
!git clone -q https://github.com/Tayyab885/Chest-Xray-Fairness-Audit.git /tmp/repo

import sys, os
os.chdir("/tmp/repo")
sys.path.insert(0, "/tmp/repo")

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
print("cpus", os.cpu_count())

In [ ]:
# Fail loudly here rather than 20 minutes into an epoch.
from src.config import load_config

cfg = load_config("configs/kaggle.yaml")
cfg["results_dir"] = f"/kaggle/working/results/{RUN}"

for f in [cfg["csv_name"], cfg["train_val_list"], cfg["test_list"]]:
    p = os.path.join(cfg["data_dir"], f)
    assert os.path.exists(p), f"MISSING {p} - is the nih-chest-xrays/data input attached?"
    print("ok", p)

img_dirs = sorted(d for d in os.listdir(cfg["data_dir"]) if d.startswith("images_"))
print("image dirs:", len(img_dirs), img_dirs[:3], "...")
assert torch.cuda.is_available(), "No GPU. Settings > Accelerator > GPU, then re-run."

In [ ]:
import logging, time
logging.basicConfig(level=logging.INFO, force=True)

from src.train import run_training
from src.mitigate import run_mitigation

t0 = time.time()
if RUN == "baseline":
    ckpt = run_training(cfg, tag="baseline")
elif RUN == "mitigated":
    ckpt = run_mitigation(cfg, group_key="sex")
else:
    raise ValueError(f"unknown RUN: {RUN}")

print(f"checkpoint: {ckpt} | {(time.time() - t0) / 3600:.2f}h")

In [ ]:
from src.run_audit import audit

audit(cfg, ckpt)

import pandas as pd
for name in ["auroc", "fairness_sex", "fairness_age", "eqodds"]:
    print(f"\n===== {name} =====")
    print(pd.read_csv(f"{cfg['results_dir']}/{name}.csv", index_col=0))

In [ ]:
# Everything below /kaggle/working is downloadable from the commit's Output tab.
for dp, _, files in os.walk("/kaggle/working"):
    for f in files:
        p = os.path.join(dp, f)
        print(f"{os.path.getsize(p) / 1e6:8.2f} MB  {p}")